# Impala67 – Handschrift-Training in Google Colab

1. Oben **Laufzeit → Laufzeittyp ändern → T4 GPU** wählen.
2. **Laufzeit → Alle ausführen**.
3. Beim ersten Mal: Google Drive freigeben und `benchmark_test_strokes.json` hochladen (liegt lokal in `handwriting/data/`).
4. BRUSH-Datensatz (~570 MB ZIP, 170 Schreiber) wird beim ersten Mal automatisch ins Drive geladen und dort als schneller Cache gespeichert.

Alles Wichtige (Zwischenstände, MathWriting-Cache, BRUSH-Cache, fertiges Modell, Log) landet in Google Drive unter `Impala67-Handschrift/`.
Bricht die Sitzung ab: einfach erneut **Alle ausführen** – das Training macht beim letzten Zwischenstand weiter.
Neues Training von vorn: `Impala67-Handschrift/checkpoints/last.pt` in Drive löschen.

**Feintuning auf eigene Handschrift** (Standard, `FEINTUNING = True`): `my_handwriting_samples.json` aus dem Abschreib-Modus nach `Impala67-Handschrift/` in Drive legen. Startet vom besseren aus `checkpoints/best.pt` und `last.pt` und speichert nach `checkpoints-feintuning/`.

In [ ]:
import os, subprocess, shutil, hashlib, glob
import torch
assert torch.cuda.is_available(), 'Keine GPU! Laufzeit -> Laufzeittyp ändern -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Impala67-Handschrift'
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)

In [ ]:
REPO = '/content/Impala67'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --depth 1 https://github.com/prinzpuma/Impala67.git {REPO}
!pip install -q onnx onnxruntime onnxscript scikit-image pillow gdown
HW = f'{REPO}/handwriting'
os.makedirs(f'{HW}/data/mw_download', exist_ok=True)
if os.path.exists(f'{DRIVE}/code.zip'):
    !unzip -o -q {DRIVE}/code.zip -d {HW}
    print('Aktualisierter Code aus Drive/code.zip übernommen')


In [ ]:
# Eigene Testzeilen (nur zum Messen, nie zum Trainieren)
bench_drive = f'{DRIVE}/benchmark_test_strokes.json'
if not os.path.exists(bench_drive):
    from google.colab import files
    print('Bitte handwriting/data/benchmark_test_strokes.json hochladen')
    up = files.upload()
    with open(bench_drive, 'wb') as f:
        f.write(next(iter(up.values())))
shutil.copy(bench_drive, f'{HW}/data/benchmark_test_strokes.json')
print('Benchmark bereit')

In [ ]:
# Google MathWriting (CC BY 4.0). Liegt der fertige Cache in Drive, entfällt der 3-GB-Download.
caches = glob.glob(f'{DRIVE}/mathwriting-2024-*-v4.cache.pkl')  # muss zu CACHE_VERSION in mathwriting_loader.py passen
for c in caches:
    shutil.copy(c, f'{HW}/data/mw_download/')

# Der test-Teil (fremde Schreiber, nur zum Messen) hat einen eigenen Cache; fehlt er, einmalig neu laden
has_test = any('-test-' in c for c in caches)
if not (caches and has_test) and not os.path.isdir(f'{HW}/data/mw_download/mathwriting-2024'):
    tgz = '/content/mathwriting-2024.tgz'
    !wget -q -c -O {tgz} https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz
    md5 = hashlib.md5()
    with open(tgz, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            md5.update(chunk)
    assert md5.hexdigest() == 'f2d59c44a545347a5f67ac70fef7a13d', 'Download beschädigt – Zelle erneut ausführen'
    !tar -xzf {tgz} -C {HW}/data/mw_download
    os.remove(tgz)

# Caches bauen (einmalig einige Minuten) und in Drive sichern
!cd {HW} && python -c "from mathwriting_loader import MathWritingDataset as M; M(); M(split='test')"
for c in glob.glob(f'{HW}/data/mw_download/*.cache.pkl'):
    if not os.path.exists(f'{DRIVE}/{os.path.basename(c)}'):
        shutil.copy(c, DRIVE)
print('MathWriting bereit')

In [ ]:
# BRUSH-Datensatz (Brown University, 170 Schreiber, ~570 MB ZIP).
# Liegt der fertige Cache in Drive, entfällt das Entpacken der 27.000 Dateien.
brush_caches = glob.glob(f'{DRIVE}/brush-*-v1.cache.pkl')
os.makedirs(f'{HW}/data/brush', exist_ok=True)
for c in brush_caches:
    shutil.copy(c, f'{HW}/data/brush/')

if not brush_caches and not os.path.isdir(f'{HW}/data/brush/writers'):
    brush_zip_drive = f'{DRIVE}/original_BRUSH.zip'
    if not os.path.exists(brush_zip_drive):
        print('Lade BRUSH ZIP einmalig nach Google Drive (~570 MB)...')
        import gdown
        gdown.download(id='1pfUKaYkFu8HpX4f5rlg0spgTk2wwbKzP', output=brush_zip_drive, quiet=False)
    print('Entpacke BRUSH ZIP in Laufzeit...')
    !unzip -o -q {brush_zip_drive} -d {HW}/data/brush/
    print('Erzeuge BRUSH Caches (einmalig für Train & Val Split)...')
    !cd {HW} && python -c "from brush_loader import BrushDataset; BrushDataset(split='train'); BrushDataset(split='val')"
    for c in glob.glob(f'{HW}/data/brush/*.cache.pkl'):
        if not os.path.exists(f'{DRIVE}/{os.path.basename(c)}'):
            shutil.copy(c, DRIVE)
print('BRUSH bereit')

In [ ]:
# Eigene Handschrift aus dem Abschreib-Modus (my_handwriting_samples.json, nur in Drive – nie auf GitHub).
# Messzeilen darin werden nur zum Messen benutzt, nie zum Trainieren.
FEINTUNING = True  # False = Basistraining von vorn (alte Variante)
samples_drive = f'{DRIVE}/my_handwriting_samples.json'
if FEINTUNING and not os.path.exists(samples_drive):
    from google.colab import files
    print('Bitte my_handwriting_samples.json hochladen (Export aus dem Abschreib-Modus)')
    up = files.upload()
    with open(samples_drive, 'wb') as f:
        f.write(next(iter(up.values())))
SAMPLES = f'{HW}/data/my_handwriting_samples.json'
if os.path.exists(samples_drive):
    shutil.copy(samples_drive, SAMPLES)

# Basis fürs Feintuning: best.pt oder last.pt aus dem letzten Lauf, gewählt nach Auswahl-CER (nie nach TEST)
if FEINTUNING:
    kandidaten = [p for p in (f'{DRIVE}/checkpoints/best.pt', f'{DRIVE}/checkpoints/last.pt') if os.path.exists(p)]
    assert kandidaten, 'Kein Checkpoint in Drive/checkpoints – erst ein Basistraining laufen lassen'
    !cd {HW} && python evaluate_checkpoints.py --samples {SAMPLES} --write-best /content/basis.txt {' '.join(kandidaten)}
    BASIS = open('/content/basis.txt').read().strip()


In [ ]:
# Training. Zwischenstände gehen nach Drive (last.pt jede Epoche, best.pt bei Verbesserung).
# Feintuning schreibt in einen eigenen Ordner, damit die Basis-Checkpoints unangetastet bleiben.
ckpt_dir = f'{DRIVE}/checkpoints-feintuning' if FEINTUNING else f'{DRIVE}/checkpoints'
args = ['python', 'train.py', '--samples', SAMPLES]
if FEINTUNING:
    args += ['--finetune', BASIS, '--epochs', '8', '--eval-every', '1', '--train-size', '20000']
env = dict(os.environ, HW_CHECKPOINT_DIR=ckpt_dir, PYTHONUNBUFFERED='1')
proc = subprocess.Popen(args, cwd=HW, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(f'{DRIVE}/train.log', 'a') as log:
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
assert proc.wait() == 0, 'Training mit Fehler beendet – siehe Ausgabe oben'


In [ ]:
# Vergleich Basis vs. Feintuning: "fremde Schreiber" zeigt, ob das Modell für andere schlechter wurde
if FEINTUNING:
    !cd {HW} && python evaluate_checkpoints.py --samples {SAMPLES} {BASIS} {ckpt_dir}/best.pt

# Fertiges Modell nach Drive kopieren und herunterladen
shutil.copy(f'{HW}/model.onnx', f'{DRIVE}/model.onnx')
print('Modell in Drive gespeichert:', f'{DRIVE}/model.onnx')
from google.colab import files
files.download(f'{HW}/model.onnx')